# 02 — Đẩy dữ liệu sạch lên SQL Server

**Mục tiêu:** Đọc `credit_risk_data_clean.csv` → tạo bảng `credit_risk_data_clean` trong database `data_Hangfolio`.

## 1. Test kết nối

In [ ]:
import sys
sys.path.append('../src')   # để import được db_connection.py trong thư mục src/
from db_connection import get_engine
engine = get_engine()

# Thử kết nối
with engine.connect() as conn:
    print('Kết nối thành công tới SQL Server')

## 2. Đọc dữ liệu sạch & đẩy lên SQL Server

`to_sql` tự động tạo bảng từ DataFrame. 
- `if_exists='replace'`: nếu bảng đã tồn tại thì ghi đè (tiện khi chạy lại nhiều lần)
- `index=False`: không đẩy cột index của pandas vào (không cần thiết)
- `chunksize=1000`: đẩy theo lô 1000 dòng, tránh nghẽn với dữ liệu lớn

In [4]:
import pandas as pd

# Đọc dữ liệu đã làm sạch
df = pd.read_csv('../Data/processed/credit_risk_data_clean.csv')
print('Dữ liệu sạch:', df.shape)

# Đẩy lên SQL Server thành bảng 'credit_risk_data_clean'
df.to_sql(
    'credit_risk_data_clean',
    con=engine,
    if_exists='replace',
    index=False,
    chunksize=1000
)
print('Đã đẩy', len(df), 'dòng lên bảng credit_risk_data_clean')

Dữ liệu sạch: (32574, 29)
Đã đẩy 32574 dòng lên bảng credit_risk_data_clean


## 3. Query thử từ SQL Server

In [5]:
# Đọc lại từ SQL Server để xác nhận dữ liệu đã lên đúng
check = pd.read_sql('SELECT COUNT(*) AS so_dong FROM credit_risk_data_clean', engine)
print(check)

# Xem thử 5 dòng đầu từ database
pd.read_sql('SELECT TOP 5 client_ID, person_age, loan_grade, loan_status FROM credit_risk_data_clean', engine)

   so_dong
0    32574


,client_ID,person_age,loan_grade,loan_status
0,CUST_00002,21,B,0
1,CUST_00003,25,C,1
2,CUST_00004,23,C,1
3,CUST_00005,24,C,1
4,CUST_00006,21,A,1


## 4. Đẩy bảng scored lên SQL Server
Section này chạy SAU khi `03_eda.ipynb` đã xuất `scored.csv` (data + risk_flags, segment, các cột band từ EDA).                
Bảng này là nguồn cho queries.sql và Power BI. Bảng clean ở section 2 giữ vai trò staging.

In [1]:
df_scored = pd.read_csv('../Data/processed/scored.csv')
print('Scored:', df_scored.shape)   # kỳ vọng ~32,574 dòng, ~37 cột

df_scored.to_sql('loans_scored', con=engine, if_exists='replace', index=False, chunksize=1000)

check = pd.read_sql('''
    SELECT segment, COUNT(*) AS n, AVG(CAST(loan_status AS FLOAT)) AS default_rate
    FROM loans_scored GROUP BY segment
''', engine)
print(check)

NameError: name 'pd' is not defined